#########    All "our" in the following code refers to Joint Cabernet.

In [1]:
import pandas as pd
import numpy as np
from itertools import chain

# Hi-C utilities imports:
import cooler
import bioframe
import cooltools
from cooltools.lib.numutils import fill_diag
from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.2'):
    raise AssertionError("tutorials rely on cooltools version 0.5.2 or higher,"+
                         "please check your cooltools version and update to the latest")

# Visualization imports:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib.patches as patches
from matplotlib.ticker import EngFormatter

# helper functions for plotting
bp_formatter = EngFormatter('b')
def format_ticks(ax, x=True, y=True, rotate=True):
    """format ticks with genomic coordinates as human readable"""
    if y:
        ax.yaxis.set_major_formatter(bp_formatter)
    if x:
        ax.xaxis.set_major_formatter(bp_formatter)
        ax.xaxis.tick_bottom()
    if rotate:
        ax.tick_params(axis='x',rotation=45)

import datetime
from tqdm.notebook import tqdm

In [ ]:
# "indir" is a custom input path, and "outdir" is a custom output path.
# indir=""
# outdir=""

In [ ]:
with open("../../../input/01-youth/subclass_order_for_integration_with_zeng.txt", "rt") as f:
    subclass_list=f.read().split("\n")[:-1]
subclass_list=[i for i in subclass_list if "NN" not in i]

csv_path = "../../../input/01-youth/subclass_our_liu.csv"
df = pd.read_csv(csv_path, sep=',')
df = df.iloc[:len(subclass_list),:]

In [ ]:
df["dots_path"]=[f"{indir}/{subclass_}.tsv" for subclass_ in df["subclass_liu"]]

In [12]:
df.head()

,subclass_our,subclass_liu,dots_path
0,L2/3 IT CTX Glut,L2_3_IT_CTX_Glut,/share/home/renlh/ll/DMR/24_distalDHMR_3Cdots_...
1,L4/5 IT CTX Glut,L4_5_IT_CTX_Glut,/share/home/renlh/ll/DMR/24_distalDHMR_3Cdots_...
2,L5 IT CTX Glut,L5_IT_CTX_Glut,/share/home/renlh/ll/DMR/24_distalDHMR_3Cdots_...
3,L2/3 IT RSP Glut,L2_3_IT_RSP_Glut,/share/home/renlh/ll/DMR/24_distalDHMR_3Cdots_...
4,L4 RSP-ACA Glut,L4_RSP-ACA_Glut,/share/home/renlh/ll/DMR/24_distalDHMR_3Cdots_...


In [19]:
mm10_chromsizes = bioframe.fetch_chromsizes('mm10')

In [21]:
np.ceil(mm10_chromsizes["chr1"]/10_000)

19548.0

In [23]:
3120000/10_000

312.0

In [24]:
np.ceil(mm10_chromsizes["chr2"]/10_000)

18212.0

In [38]:
chrom_list=["chr"+str(i) for i in range(1,20)]

In [32]:
start_bin_index={"chr"+str(chrom_number) : int(np.sum([np.ceil(mm10_chromsizes["chr"+str(i)]/10_000) for i in range(1, chrom_number)])) for chrom_number in range(1,20)}

In [33]:
start_bin_index

{'chr1': 0,
 'chr2': 19548,
 'chr3': 37760,
 'chr4': 53764,
 'chr5': 69415,
 'chr6': 84599,
 'chr7': 99573,
 'chr8': 114118,
 'chr9': 127059,
 'chr10': 139519,
 'chr11': 152589,
 'chr12': 164798,
 'chr13': 176811,
 'chr14': 188854,
 'chr15': 201345,
 'chr16': 211750,
 'chr17': 221571,
 'chr18': 231070,
 'chr19': 240141}

In [15]:
temp_dots_df=pd.read_csv(df.iloc[0]["dots_path"], sep="\t")

In [44]:
temp_dots_df=temp_dots_df[temp_dots_df["chrom1"].apply(lambda x : x in chrom_list)]

In [45]:
temp_dots_df.groupby("chrom1").head(2)["chrom1"].values

array(['chr1', 'chr1', 'chr2', 'chr2', 'chr3', 'chr3', 'chr4', 'chr4',
       'chr5', 'chr5', 'chr6', 'chr6', 'chr7', 'chr7', 'chr8', 'chr8',
       'chr9', 'chr9', 'chr10', 'chr10', 'chr11', 'chr11', 'chr12',
       'chr12', 'chr13', 'chr13', 'chr14', 'chr14', 'chr15', 'chr15',
       'chr16', 'chr16', 'chr17', 'chr17', 'chr18', 'chr18', 'chr19',
       'chr19'], dtype=object)

In [47]:
np.all(temp_dots_df.groupby("chrom1").head(300)["bin1"] == \
temp_dots_df.groupby("chrom1").head(300)["start1"]/10_000 + \
[start_bin_index[tt] for tt in temp_dots_df.groupby("chrom1").head(300)["chrom1"].values])

True

In [51]:
temp_dots_df.iloc[0,2] in {3080000}

True

In [ ]:
genes_df = pd.read_csv('../../../input/reference_genome/Genebody.mm10.bed', sep='\t', header=None, names=['chromosome', 'start', 'end', 'gene_id', 'gene_name', 'strand'])

In [14]:
genes_df

,chromosome,start,end,gene_id,gene_name,strand
0,chr1,3073253,3074322,ENSMUSG00000102693.1,RP23-271O17.1,+
1,chr1,3102016,3102125,ENSMUSG00000064842.1,Gm26206,+
2,chr1,3205901,3671498,ENSMUSG00000051951.5,Xkr4,-
3,chr1,3252757,3253236,ENSMUSG00000102851.1,RP23-317L18.1,+
4,chr1,3365731,3368549,ENSMUSG00000103377.1,RP23-317L18.4,-
...,...,...,...,...,...,...
54141,chrM,13552,14070,ENSMUSG00000064368.1,mt-Nd6,-
54142,chrM,14071,14139,ENSMUSG00000064369.1,mt-Te,-
54143,chrM,14145,15288,ENSMUSG00000064370.1,mt-Cytb,+
54144,chrM,15289,15355,ENSMUSG00000064371.1,mt-Tt,+


In [52]:
genes_df=genes_df[genes_df["chromosome"].apply(lambda x: x in chrom_list)]

In [53]:
genes_df

,chromosome,start,end,gene_id,gene_name,strand
0,chr1,3073253,3074322,ENSMUSG00000102693.1,RP23-271O17.1,+
1,chr1,3102016,3102125,ENSMUSG00000064842.1,Gm26206,+
2,chr1,3205901,3671498,ENSMUSG00000051951.5,Xkr4,-
3,chr1,3252757,3253236,ENSMUSG00000102851.1,RP23-317L18.1,+
4,chr1,3365731,3368549,ENSMUSG00000103377.1,RP23-317L18.4,-
...,...,...,...,...,...,...
49920,chr19,61164365,61164481,ENSMUSG00000070263.1,Gm22365,-
49921,chr19,61174686,61176309,ENSMUSG00000094649.1,RP24-318N5.1,-
49922,chr19,61183890,61183955,ENSMUSG00000069475.1,Gm6020,+
49923,chr19,61224402,61228418,ENSMUSG00000059326.6,Csf2ra,-


In [55]:
genes_bin_index=set()
for _, i in tqdm(genes_df.iterrows(), total=len(genes_df)):
    chrom_=i["chromosome"]
    start_=int(i["start"])
    end_=int(i["end"])
    start_bin_=int(start_bin_index[chrom_]+np.floor(start_/10_000))
    end_bin_=int(start_bin_index[chrom_]+np.floor(end_/10_000))
    for j in range(start_bin_, end_bin_+1):
        genes_bin_index.add(j)

  0%|          | 0/49925 [00:00<?, ?it/s]

In [56]:
len(genes_bin_index)

139623

In [ ]:
RNA_origin="our"

with open("../../../input/01-youth/subclass_order_for_integration_with_zeng.txt", "rt") as f:
    subclass_list=f.read().split("\n")[:-1]
subclass_list=[i for i in subclass_list if "NN" not in i]

df_RNA=pd.read_csv(f"{indir}/02-{RNA_origin}_subclass_mean_dat_final.csv",index_col=0).T

max_threshold_=0.2
cv_threshold_=0.5
df_RNA_selected=df_RNA.loc[subclass_list,:]
df_RNA_selected=df_RNA_selected.loc[subclass_list,
    (df_RNA_selected.max(axis=0)>max_threshold_) &
    (df_RNA_selected.apply(lambda col: col.std() / col.mean(), axis=0)>cv_threshold_)
            ]

/tmp/ipykernel_33068/8151829.py:14: RuntimeWarning: invalid value encountered in scalar divide
  (df_RNA_selected.apply(lambda col: col.std() / col.mean(), axis=0)>cv_threshold_)


In [63]:
selected_genes_bin_index=set()

for _, i in tqdm(genes_df.iterrows(), total=len(genes_df)):
    if i["gene_id"] not in df_RNA_selected.columns:
        continue
    chrom_=i["chromosome"]
    start_=int(i["start"])
    end_=int(i["end"])
    start_bin_=int(start_bin_index[chrom_]+np.floor(start_/10_000))
    end_bin_=int(start_bin_index[chrom_]+np.floor(end_/10_000))
    for j in range(start_bin_, end_bin_+1):
        selected_genes_bin_index.add(j)

  0%|          | 0/49925 [00:00<?, ?it/s]

In [64]:
len(selected_genes_bin_index)

27788

In [120]:
# now select for the dots with one end in the intergenic region (i.e. not in genes_bin_index),
# and the other end in the selected_genes_bin_index.
# One thing is that every subclasses can yield some loops fulfilling the above conditions, check it out.
all_wanted_dots_df_list=[]

for _, i in df.iterrows():
    subclass_dots_df=pd.read_csv(i["dots_path"], sep="\t")
    # remove chrX, chrY, chrM
    subclass_dots_df=subclass_dots_df[subclass_dots_df["chrom1"].apply(lambda x : x in chrom_list)]
    subclass_dots_df["subclass"]=i["subclass_our"]
    # record the status of bin1 and bin2
    subclass_dots_df["bin1_intergenic"]=subclass_dots_df["bin1"].apply(lambda x : x not in genes_bin_index)
    subclass_dots_df["bin1_selected"]=subclass_dots_df["bin1"].apply(lambda x : x in selected_genes_bin_index)
    subclass_dots_df["bin2_intergenic"]=subclass_dots_df["bin2"].apply(lambda x : x not in genes_bin_index)
    subclass_dots_df["bin2_selected"]=subclass_dots_df["bin2"].apply(lambda x : x in selected_genes_bin_index)
    subclass_dots_df["bin_status"]=subclass_dots_df[["bin1_intergenic", "bin1_selected", "bin2_intergenic", "bin2_selected"]].astype(int).astype(str).agg("".join, axis=1)
    # get wanted dots
    subclass_wanted_dots_df=subclass_dots_df[subclass_dots_df["bin_status"].apply(lambda x : x in ["0110", "1001"])]
    print(i["subclass_our"], subclass_wanted_dots_df.shape)
    all_wanted_dots_df_list.append(subclass_wanted_dots_df)

all_wanted_dots_df=pd.concat(all_wanted_dots_df_list)

L2/3 IT CTX Glut (33992, 19)
L4/5 IT CTX Glut (34113, 19)
L5 IT CTX Glut (37377, 19)
L2/3 IT RSP Glut (40404, 19)
L4 RSP-ACA Glut (42739, 19)
L5 ET CTX Glut (38476, 19)
SUB-ProS Glut (41336, 19)
CA1-ProS Glut (36835, 19)
CA3 Glut (38546, 19)
CLA-EPd-CTX Car3 Glut (40319, 19)
L5 NP CTX Glut (39859, 19)
L6 CT CTX Glut (34572, 19)
DG Glut (33861, 19)
OB Eomes Ms4a15 Glut (43455, 19)
OB-in Frmd7 Gaba (38132, 19)
OB-out Frmd7 Gaba (43112, 19)
OB Dopa-Gaba (43563, 19)
OB-STR-CTX Inh IMN (39261, 19)
Sncg Gaba (41138, 19)
Lamp5 Gaba (38382, 19)
Pvalb Gaba (36197, 19)
Sst Gaba (36427, 19)
STR D1 Gaba (34819, 19)
STR D2 Gaba (35138, 19)
ACB-BST-FS D1 Gaba (42715, 19)


In [122]:
all_wanted_dots_df

,chrom1,start1,end1,chrom2,start2,end2,bin1,bin2,kernel_id,iteration,score,pvalue,qvalue,subclass,bin1_intergenic,bin1_selected,bin2_intergenic,bin2_selected,bin_status
5,chr1,3050000,3060000,chr1,5040000,5050000,305,504,0,0,0.482867,1.000000e-10,1.000000e-10,L2/3 IT CTX Glut,True,False,False,True,1001
22,chr1,3110000,3120000,chr1,6740000,6750000,311,674,0,0,0.363741,6.510000e-08,6.850000e-08,L2/3 IT CTX Glut,True,False,False,True,1001
23,chr1,3110000,3120000,chr1,6790000,6800000,311,679,0,0,0.417278,3.000000e-10,4.000000e-10,L2/3 IT CTX Glut,True,False,False,True,1001
107,chr1,3690000,3700000,chr1,6520000,6530000,369,652,0,0,0.316687,2.920000e-08,3.150000e-08,L2/3 IT CTX Glut,True,False,False,True,1001
115,chr1,3730000,3740000,chr1,6680000,6690000,373,668,0,0,0.483908,0.000000e+00,0.000000e+00,L2/3 IT CTX Glut,True,False,False,True,1001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436745,chr19,59760000,59770000,chr19,60950000,60960000,246117,246236,0,0,0.431532,8.000000e-10,9.000000e-10,ACB-BST-FS D1 Gaba,True,False,False,True,1001
436751,chr19,59840000,59850000,chr19,60950000,60960000,246125,246236,0,0,0.534147,0.000000e+00,0.000000e+00,ACB-BST-FS D1 Gaba,True,False,False,True,1001
436767,chr19,60090000,60100000,chr19,61010000,61020000,246150,246242,0,0,0.330801,4.860000e-08,5.060000e-08,ACB-BST-FS D1 Gaba,True,False,False,True,1001
436773,chr19,60250000,60260000,chr19,60980000,60990000,246166,246239,0,0,0.483798,0.000000e+00,0.000000e+00,ACB-BST-FS D1 Gaba,True,False,False,True,1001


In [ ]:
all_wanted_dots_df.to_csv(f"{outdir}/intergenic_to_selectedGenes_dots.tsv", sep="\t", index=False)